# 단일 TODO 분할기(splitter) 변경 전후 비교

**날짜:** 2026-06-28  ·  **대상:** `agents/todo_creation/todo/nodes/task_splitter.py`

qwen→EXAONE 전환 후 "복잡한 입력이 안 나뉨" 문제를 고친 3가지 변경을 **측정 가능하게** 비교한다.

| # | 변경 | 전(OLD) | 후(NEW) |
|---|---|---|---|
| 1 | 압축률 게이트 | `_is_low_information` 로 LLM 전 차단 | **삭제** |
| 2 | 그라운딩 | 글자 2-gram 겹침 | **어절 접두 매칭** |
| 3 | 중복 | 없음 | **(title,due_date) dedup** |

방법: **NEW 로직은 실제 모듈에서 import**(현행 코드 그대로 측정), **OLD 로직은 참조용으로 재현**.
입력은 실제 RunPod EXAONE(adapter=base) 라이브 관측 출력을 고정값으로 사용 → GPU 없이 재현 가능.
맨 아래 셀로 실제 EXAONE 재호출(.env 필요)도 선택 가능.

## 0. 셋업

In [1]:
import sys, os, zlib
from datetime import date
sys.path.insert(0, os.path.abspath('..'))  # notebooks/ -> repo root

# === NEW: 실제 현행 코드 import ===
from agents.todo_creation.todo.nodes.task_splitter import (
    _is_grounded as new_is_grounded,
    _repair_title,
)
print('현행 코드 import 완료')

현행 코드 import 완료


## 1. 변경 전(OLD) 로직 재현 — 참조용

In [2]:
# --- OLD 1: 압축률 게이트 (삭제됨) ---
_LOW_INFO_THRESHOLD, _LOW_INFO_MIN_BYTES, _LOW_INFO_MAX_BYTES = 0.75, 30, 400
def old_is_low_information(text: str) -> bool:
    raw = text.encode('utf-8')
    if not (_LOW_INFO_MIN_BYTES <= len(raw) <= _LOW_INFO_MAX_BYTES):
        return False
    return len(zlib.compress(raw, 6)) / len(raw) < _LOW_INFO_THRESHOLD

# --- OLD 2: 글자 2-gram 그라운딩 (교체됨) ---
def _char_bigrams(text: str) -> set:
    s = ''.join(text.split())
    return {s[i:i+2] for i in range(len(s) - 1)}
def old_is_grounded(title: str, prompt: str) -> bool:
    tb = _char_bigrams(title)
    if not tb:
        return True
    return bool(tb & _char_bigrams(prompt))
print('OLD 로직 재현 완료')

OLD 로직 재현 완료


## 2. 두 파이프라인 정의

task 는 `(title, due_date)` 튜플로 표현. repair 는 두 버전 공통(변경 없음)이라 NEW 모듈 것을 같이 쓴다.

In [3]:
def run_old(prompt, tasks):
    if old_is_low_information(prompt):              # 게이트
        return ('out_of_scope', [])
    repaired = [(_repair_title(t, prompt), d) for t, d in tasks]
    grounded = [(t, d) for t, d in repaired if old_is_grounded(t, prompt)]
    if not grounded:
        return ('out_of_scope', [])
    return ('plan', grounded)                       # dedup 없음

def run_new(prompt, tasks):
    # 게이트 없음
    repaired = [(_repair_title(t, prompt), d) for t, d in tasks]
    grounded = [(t, d) for t, d in repaired if new_is_grounded(t, prompt)]
    if not grounded:
        return ('out_of_scope', [])
    deduped = list({(t, d): (t, d) for t, d in grounded}.values())  # dedup
    return ('plan', deduped)

## 3. 데이터셋 — 실제 EXAONE 라이브 관측 출력

케이스 3의 모델 출력은 EXAONE 이 실제로 깬 `두啭iku 먹기` 원형을 그대로 넣어 repair 를 양쪽에서 실행시킨다.

In [4]:
D = date(2026, 6, 28)
CASES = [
    {'name': '1. 반복', 'input': '밥을 먹고 밥을 먹어야지 밥을 먹어야해',
     'model_tasks': [('밥 먹기', D), ('밥 먹기', D), ('밥 먹기', D)]},
    {'name': '2. 복잡',
     'input': ('오뚜기 밥을 먹어야 되고 반찬도 필요하니까 반찬가게 가서 반찬을 사와야 되겠다. '
               '쌀과자도 후식으로 먹어야 되는데 다 떨어졌으니까 마트 들려서 쌀과자도 사야겠어'),
     'model_tasks': [('오뚜기 밥 먹기', D), ('반찬 사기', D), ('쌀과자 사기', D)]},
    {'name': '3. 희귀어', 'input': '헬스장 갔다가 두쫀쿠 먹고 엄마 보러 가야지',
     'model_tasks': [('헬스장 가기', D), ('두啭iku 먹기', D), ('엄마 보러 가기', D)]},
]
print(f'{len(CASES)} 케이스')

3 케이스


## 4. 전후 비교 (메인 결과)

In [5]:
def fmt(tasks):
    return ' / '.join(t for t, _ in tasks) if tasks else '—'

print(f"{'케이스':<8} {'OLD intent':<12} {'OLD tasks':<26} {'NEW intent':<12} {'NEW tasks':<26} 판정")
print('-' * 100)
changed = 0
for c in CASES:
    oi, ot = run_old(c['input'], c['model_tasks'])
    ni, nt = run_new(c['input'], c['model_tasks'])
    same = (oi, ot) == (ni, nt)
    changed += (not same)
    verdict = '동일' if same else '✅ 개선'
    print(f"{c['name']:<8} {oi:<12} {fmt(ot):<26} {ni:<12} {fmt(nt):<26} {verdict}")
print('-' * 100)
print(f'달라진 케이스: {changed}/{len(CASES)}')

케이스      OLD intent   OLD tasks                  NEW intent   NEW tasks                  판정
----------------------------------------------------------------------------------------------------
1. 반복    out_of_scope —                          plan         밥 먹기                       ✅ 개선
2. 복잡    out_of_scope —                          plan         오뚜기 밥 먹기 / 반찬 사기 / 쌀과자 사기  ✅ 개선
3. 희귀어   plan         헬스장 가기 / 두쫀쿠 먹기 / 엄마 보러 가기 plan         헬스장 가기 / 두쫀쿠 먹기 / 엄마 보러 가기 동일
----------------------------------------------------------------------------------------------------
달라진 케이스: 2/3


## 5. 지표 ①: 그라운딩 정확도

정상 task 를 잘못 떨구는 **false-drop** 과 환각을 통과시키는 **missed** 를 센다(둘 다 낮을수록 좋음).

In [6]:
LEGIT = [  # 전부 통과해야 정상 (정규화로 조사가 떨어진 케이스 포함)
    ('밥 먹기', '밥을 먹어야지'), ('숙제 하기', '숙제를 해야해'),
    ('물 마시기', '물을 마시자'), ('청소 하기', '방을 청소해야지'),
    ('과제 제출', '과제를 제출하기'), ('약 먹기', '약을 먹어야지'),
]
HALLUC = [  # 전부 드롭해야 정상 (입력에 없는 환각)
    ('토익 공부', '오늘 코테 발표'), ('여행 가기', '회의 준비하고 보고서 작성'),
]
def score(fn):
    false_drop = sum(1 for t, p in LEGIT if not fn(t, p))
    missed = sum(1 for t, p in HALLUC if fn(t, p))
    return false_drop, missed

print(f"{'그라운딩':<16}{'정상 false-drop':>16}{'환각 missed':>14}")
for nm, f in [('OLD (bigram)', old_is_grounded), ('NEW (prefix)', new_is_grounded)]:
    fd, mi = score(f)
    print(f'{nm:<16}{fd:>12}/{len(LEGIT)}{mi:>10}/{len(HALLUC)}')
print()
print('상세(정상인데 드롭되는 것):')
for t, p in LEGIT:
    o = '' if old_is_grounded(t, p) else 'OLD-DROP'
    n = '' if new_is_grounded(t, p) else 'NEW-DROP'
    if o or n:
        print(f'  {t!r:14} <= {p!r:18} {o} {n}')

그라운딩               정상 false-drop     환각 missed
OLD (bigram)               2/6         0/2
NEW (prefix)               0/6         0/2

상세(정상인데 드롭되는 것):
  '밥 먹기'         <= '밥을 먹어야지'          OLD-DROP 
  '약 먹기'         <= '약을 먹어야지'          OLD-DROP 


## 6. 지표 ②: 압축률 게이트 false-positive

정상 입력이 0.75 임계값에 걸려 LLM 호출 전에 버려지는지 확인. NEW 는 게이트가 없어 전부 LLM 도달.

In [7]:
SAMPLES = {
    '정상 복잡(케이스2)': CASES[1]['input'],
    '정상 짧음': '장보고 운동해야지 내일 친구 만나기',
    'degenerate 반복': '건강하고 건강하며 건강한데 또 건강했다가 건강하려다가',
    '스팸': 'ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ',
}
print(f"{'입력':<22}{'압축률':>8}   OLD 게이트            NEW")
print('-' * 64)
for k, t in SAMPLES.items():
    raw = t.encode('utf-8')
    r = len(zlib.compress(raw, 6)) / len(raw)
    old = 'BLOCK -> out_of_scope' if old_is_low_information(t) else 'pass'
    print(f'{k:<22}{r:>8.3f}   {old:<22}LLM 도달')
print()
print('핵심: degenerate(약 0.67)와 정상 복잡(약 0.72)이 겹쳐 임계값으로 분리 불가 → 삭제')

입력                         압축률   OLD 게이트            NEW
----------------------------------------------------------------
정상 복잡(케이스2)              0.721   BLOCK -> out_of_scope LLM 도달
정상 짧음                    1.224   pass                  LLM 도달
degenerate 반복            0.727   BLOCK -> out_of_scope LLM 도달
스팸                       0.292   BLOCK -> out_of_scope LLM 도달

핵심: degenerate(약 0.67)와 정상 복잡(약 0.72)이 겹쳐 임계값으로 분리 불가 → 삭제


## 7. (선택) 실제 EXAONE 재호출로 model_tasks 재생성

`.env` 의 `RUNPOD_PLANNER_ENDPOINT_URL`/`RUNPOD_API_KEY` 가 있으면 위 고정값 대신 실제 출력으로 갱신.
콜드스타트 시 수십 초 걸릴 수 있음. 셀을 실행하지 않으면 위 라이브 관측 고정값으로 비교가 끝난다.

In [8]:
RUN_LIVE = False  # True 로 바꾸면 실제 EXAONE 호출
if RUN_LIVE:
    import asyncio
    from adapters.todo_creation.runpod_llm import RunPodQwenLLM
    env = {}
    for line in open('../.env'):
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            env[k.strip()] = v.strip().strip('"').strip("'")
    llm = RunPodQwenLLM(endpoint_url=env['RUNPOD_PLANNER_ENDPOINT_URL'],
                        api_key=env['RUNPOD_API_KEY'], adapter='base')
    async def regen():
        for c in CASES:
            res = await llm.split_tasks(prompt=c['input'], today=D)
            c['model_tasks'] = [(t.title, t.due_date) for t in res.tasks]
            print(c['name'], '->', [t.title for t in res.tasks])
    asyncio.run(regen())
    print('\n재생성 완료 — 4~6 셀을 다시 실행해 비교하세요.')
else:
    print('RUN_LIVE=False — 라이브 관측 고정값 사용 중')

RUN_LIVE=False — 라이브 관측 고정값 사용 중


## 결론

- **케이스 1·2** 가 OLD 에선 깨졌고(1: out_of_scope, 또는 게이트 차단 / 2: 게이트가 막던 입력) NEW 에서 정상.
- **그라운딩**: OLD 는 조사로 갈린 정상 task(밥 먹기 ← 밥을)를 false-drop. NEW(접두)는 0 으로 줄이면서 환각은 그대로 차단.
- **게이트**: 정상 복잡 입력이 압축률로 차단됨. degenerate 와 겹쳐 임계값 조정 불가 → 삭제가 정답.
- **dedup**: 모델 과분해(밥 먹기 ×3)를 1개로.
- repair 는 EXAONE 도 동일하게 깨므로 유지(케이스 3 에서 양쪽 동일 동작 확인).